<a href="https://colab.research.google.com/github/trisha1maheshwari/Ollama-Data/blob/main/Ollama_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run Ollama in Colab
---

[![5aharsh/collama](https://raw.githubusercontent.com/5aharsh/collama/main/assets/banner.png)](https://github.com/5aharsh/collama)

This is an example notebook which demonstrates how to run Ollama inside a Colab instance. With this you can run pretty much any small to medium sized models offerred by Ollama for free.

For the list of available models check [models being offerred by Ollama](https://ollama.com/library).


## Before you proceed
---

Since by default the runtime type of Colab instance is CPU based, in order to use LLM models make sure to change your runtime type to T4 GPU (or better if you're a paid Colab user). This can be done by going to **Runtime > Change runtime type**.

While running your script be mindful of the resources you're using. This can be tracked at **Runtime > View resources**.

## Running the notebook
---

After configuring the runtime just run it with **Runtime > Run all**. And you can start tinkering around. This example uses [Llama 3.2](https://ollama.com/library/llama3.2) to generate a response from a prompted question using [LangChain Ollama Integration](https://python.langchain.com/docs/integrations/chat/ollama/).

## Installing Dependencies
---

1. `pciutils` is required by Ollama to detect the GPU type.
2. Installation of Ollama in the runtime instance will be taken care by `curl -fsSL https://ollama.com/install.sh | sh`




In [3]:
!sudo apt update
!sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
103 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not see

## Running Ollama
---

In order to use Ollama it needs to run as a service in background parallel to your scripts. Becasue Jupyter Notebooks is built to run code blocks in sequence this make it difficult to run two blocks at the same time. As a workaround we will create a service using subprocess in Python so it doesn't block any cell from running.

Service can be started by command `ollama serve`.

`time.sleep(5)` adds some delay to get the Ollama service up before downloading the model.

In [4]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

## Pulling Model
---

Download the LLM model using `ollama pull llama3.2`.

For other models check https://ollama.com/library

In [5]:
!ollama pull llama3.1:8b

## And that's it!
---

With this you should be able to freely play around with the models in your scripts. Following is an example using `langchain-ollama` to answer a simple prompt.

If you have a use-case that can help out others feel free to add your notebook to [Collama](https://github.com/5aharsh/collama/fork)

In [6]:
!pip install pandas yfinance ollama ddgs openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 51.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ollama]


In [7]:
import os
import json
import time
import pandas as pd
import yfinance as yf  # pip install yfinance
import ollama          # pip install ollama
from ddgs import DDGS  # pip install ddgs
from datetime import datetime

# Full element names improve search accuracy for commodity pricing queries
ELEMENT_NAMES = {
    "Li": "Lithium", "Na": "Sodium", "K": "Potassium", "Rb": "Rubidium",
    "Cs": "Cesium", "Fr": "Francium", "Be": "Beryllium", "Mg": "Magnesium",
    "Ca": "Calcium", "Sr": "Strontium", "Ba": "Barium", "Ra": "Radium",
    "Sc": "Scandium", "Ti": "Titanium", "V": "Vanadium", "Cr": "Chromium",
    "Mn": "Manganese", "Fe": "Iron", "Co": "Cobalt", "Ni": "Nickel",
    "Cu": "Copper", "Zn": "Zinc", "Y": "Yttrium", "Zr": "Zirconium",
    "Nb": "Niobium", "Mo": "Molybdenum", "Tc": "Technetium", "Ru": "Ruthenium",
    "Rh": "Rhodium", "Pd": "Palladium", "Ag": "Silver", "Cd": "Cadmium",
    "Hf": "Hafnium", "Ta": "Tantalum", "W": "Tungsten", "Re": "Rhenium",
    "Os": "Osmium", "Ir": "Iridium", "Pt": "Platinum", "Au": "Gold",
    "Hg": "Mercury", "Rf": "Rutherfordium", "Db": "Dubnium", "Sg": "Seaborgium",
    "Bh": "Bohrium", "Hs": "Hassium", "Mt": "Meitnerium", "Ds": "Darmstadtium",
    "Rg": "Roentgenium", "Cn": "Copernicium", "Al": "Aluminum", "Ga": "Gallium",
    "In": "Indium", "Sn": "Tin", "Tl": "Thallium", "Pb": "Lead",
    "Bi": "Bismuth", "Nh": "Nihonium", "Fl": "Flerovium", "Mc": "Moscovium",
    "Lv": "Livermorium", "La": "Lanthanum", "Ce": "Cerium", "Pr": "Praseodymium",
    "Nd": "Neodymium", "Pm": "Promethium", "Sm": "Samarium", "Eu": "Europium",
    "Gd": "Gadolinium", "Tb": "Terbium", "Dy": "Dysprosium", "Ho": "Holmium",
    "Er": "Erbium", "Tm": "Thulium", "Yb": "Ytterbium", "Lu": "Lutetium",
    "Th": "Thorium", "U": "Uranium"
}

# -------------------------------------------------------------------------
# STRATEGY 1: Financial APIs (yfinance) for Major Traded Commodities
# -------------------------------------------------------------------------
def get_live_market_price(symbol):
    try:
        ticker_map = {
            "Au": "GC=F",  # Gold
            "Ag": "SI=F",  # Silver
            "Cu": "HG=F",  # Copper
        }
        if symbol in ticker_map:
            ticker = yf.Ticker(ticker_map[symbol])
            hist = ticker.history(period="1d")
            if not hist.empty:
                price_per_oz_or_lb = hist['Close'].iloc[-1]
                if symbol in ["Au", "Ag"]:
                    price_per_kg = price_per_oz_or_lb / 0.0311035
                elif symbol == "Cu":
                    price_per_kg = price_per_oz_or_lb / 0.453592
                else:
                    price_per_kg = price_per_oz_or_lb
                return round(price_per_kg, 2)
    except Exception as e:
        print(f"API fetch fallback for {symbol}: {e}")
    return None

# -------------------------------------------------------------------------
# STRATEGY 2: Exact-Date Live Web Search (DDGS)
# -------------------------------------------------------------------------
def fetch_web_price_context(symbol):
    element_name = ELEMENT_NAMES.get(symbol, symbol)
    exact_date_str = datetime.now().strftime("%B %d %Y") # e.g., November 16 2026

    # Inject exact date parameters to target today's spot quotes / daily reports
    queries = [
        f"{element_name} metal spot price per kg USD {exact_date_str}",
        f"cost of {element_name} per kilogram market price {exact_date_str}",
        f"{element_name} daily commodity price {exact_date_str}"
    ]

    for query in queries:
        try:
            results = DDGS().text(query, region="us-en", max_results=3)
            if results:
                snippets = []
                for r in results:
                    title = r.get("title", "")
                    body = r.get("body", "")
                    snippets.append(f"- Source: {title}\n  Details: {body}")
                return "\n".join(snippets)
        except Exception as e:
            continue

    return f"No exact-date search results found for {exact_date_str}. Use latest available market data."

# Periodic table elements list
metals = list(ELEMENT_NAMES.keys())

current_date_str = datetime.now().strftime("%Y-%m-%d")
excel_file = f"{current_date_str}_metal_prices_web.xlsx"

# Load existing progress to support easy resumption if interrupted
results = []
processed_symbols = set()
if os.path.exists(excel_file):
    df_existing = pd.read_excel(excel_file)
    results = df_existing.to_dict(orient="records")
    processed_symbols = set(df_existing["symbol"].values)
    print(f"Resuming! Found {len(processed_symbols)} already processed metals.")

for metal in metals:
    if metal in processed_symbols:
        print(f"Skipping {metal} (already saved).")
        continue

    element_name = ELEMENT_NAMES.get(metal, metal)
    print(f"Fetching price for {metal} ({element_name}) for today ({current_date_str})...")

    # Check live market API first
    live_price = get_live_market_price(metal)
    if live_price is not None:
        item_data = {
            "symbol": metal,
            "cost_per_kg": live_price,
            "currency": "USD"
        }
        print(f"Success (via Market API): {item_data}")
        results.append(item_data)
        pd.DataFrame(results).to_excel(excel_file, index=False, engine="openpyxl")
        continue

    # Fetch live web snippets using today's exact date
    web_context = fetch_web_price_context(metal)

    # -------------------------------------------------------------------------
    # STRATEGY 3: Local Ollama Extraction with Exact-Date Context & JSON Mode
    # -------------------------------------------------------------------------
    exact_date_readable = datetime.now().strftime("%A, %B %d, %Y")
    prompt = f"""You are a metallurgical commodity pricing analyst. Today is exactly {exact_date_readable}. Extract the precise spot price per kg in USD for {element_name} (symbol: '{metal}') reflecting today's market conditions using the search results below.

Live Web Search Results (for {exact_date_readable}):
{web_context}

Rules:
1. Strongly prioritize prices matching today's exact date or the most recent market close available in the snippets.
2. Convert units if necessary: USD/lb (multiply by 2.20462), USD/metric ton (divide by 1000), or USD/troy oz (multiply by 32.1507).
3. If snippets lack a precise numeric rate for today, provide the closest reliable recent market estimate in USD/kg, or 0.0 for synthetic/radioactive elements.
4. Return ONLY a valid JSON object matching this exact schema:
{{
  "symbol": "{metal}",
  "cost_per_kg": 0.0,
  "currency": "USD"
}}"""

    success = False
    retries = 3

    while not success and retries > 0:
        try:
            response = ollama.chat(
                model="llama3.1:8b",  # Change to 'llama3.2:3b' if running on lower RAM
                messages=[{"role": "user", "content": prompt}],
                format="json",        # Enforces strict valid JSON output from Ollama
                options={"temperature": 0.0}
            )

            text = response["message"]["content"].strip()
            item_data = json.loads(text)

            cleaned_data = {
                "symbol": metal,
                "cost_per_kg": round(float(item_data.get("cost_per_kg", 0.0)), 4),
                "currency": "USD"
            }

            print(f"Success (via Web Search + Ollama): {cleaned_data}")
            results.append(cleaned_data)

            # Save incrementally to Excel
            pd.DataFrame(results).to_excel(excel_file, index=False, engine="openpyxl")
            success = True

        except Exception as e:
            print(f"Error processing {metal}: {e}")
            retries -= 1
            if retries > 0:
                print("Retrying in 3 seconds...")
                time.sleep(3)
            else:
                print(f"Skipping {metal} after multiple failures, defaulting to 0.0")
                results.append({"symbol": metal, "cost_per_kg": 0.0, "currency": "USD"})
                pd.DataFrame(results).to_excel(excel_file, index=False, engine="openpyxl")

    time.sleep(2)

print(f"\nAll done! Total records saved: {len(results)} in {excel_file}")

Fetching price for Li (Lithium) for today (2026-11-19)...
Success (via Web Search + Ollama): {'symbol': 'Li', 'cost_per_kg': 13.69, 'currency': 'USD'}
Fetching price for Na (Sodium) for today (2026-11-19)...
Success (via Web Search + Ollama): {'symbol': 'Na', 'cost_per_kg': 0.0, 'currency': 'USD'}
Fetching price for K (Potassium) for today (2026-11-19)...
Success (via Web Search + Ollama): {'symbol': 'K', 'cost_per_kg': 0.0, 'currency': 'USD'}
Fetching price for Rb (Rubidium) for today (2026-11-19)...
Success (via Web Search + Ollama): {'symbol': 'Rb', 'cost_per_kg': 0.0, 'currency': 'USD'}
Fetching price for Cs (Cesium) for today (2026-11-19)...
Success (via Web Search + Ollama): {'symbol': 'Cs', 'cost_per_kg': 0.0, 'currency': 'USD'}
Fetching price for Fr (Francium) for today (2026-11-19)...
Success (via Web Search + Ollama): {'symbol': 'Fr', 'cost_per_kg': 0.0, 'currency': 'USD'}
Fetching price for Be (Beryllium) for today (2026-11-19)...
Success (via Web Search + Ollama): {'symbol'

Success (via Web Search + Ollama): {'symbol': 'U', 'cost_per_kg': 200.0, 'currency': 'USD'}

All done! Total records saved: 78 in 2026-11-19_metal_prices_web.xlsx


In [8]:
from google.colab import files
files.download("2026-11-19_metal_prices_web.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>